
# 05 — Classificação por Faixas Etárias, Limiares e Matrizes de Confusão

Este notebook realiza a avaliação das classificações etárias produzidas pelos modelos DeepFace e MiVOLO sobre a base UTKFace.

São realizadas duas análises complementares:

1. **Classificação multiclasse:** organização das idades em quatro faixas etárias:
   - 0 a 11 anos;
   - 12 a 15 anos;
   - 16 a 17 anos;
   - 18 anos ou mais.

2. **Classificação binária por limiares:** avaliação dos resultados considerando os pontos de corte de 12, 16 e 18 anos, além do limiar complementar de 17 anos.

Para cada classificação, são elaboradas matrizes de confusão por tecnologia e partição da base, bem como resultados consolidados.

Na avaliação binária, são calculados os valores de verdadeiros positivos (TP), verdadeiros negativos (TN), falsos positivos (FP) e falsos negativos (FN), além das métricas de acurácia, taxa de verdadeiros positivos (TPR), taxa de falsos positivos (FPR), taxa de falsos negativos (FNR) e taxa de verdadeiros negativos (TNR).

As análises consideram a idade informada no nome do arquivo como idade de referência, sem pressupor que ela represente uma idade cronológica comprovada.

**Critérios metodológicos:**
- A classe positiva na classificação binária corresponde à idade de referência igual ou superior ao limiar avaliado.
- Estimativas negativas não são consideradas decisões válidas para classificação, mas permanecem preservadas nos resultados originais.
- Registros sem decisão válida são identificados e contabilizados separadamente.
- A avaliação considera os resultados individuais de cada partição e o conjunto consolidado da base UTKFace.


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


In [ ]:

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# ------------------------------------------------------------
# Diretórios dos resultados
# ------------------------------------------------------------

PASTA_RESULTADOS_DEEPFACE = (
    "/content/drive/MyDrive/TCC-Estimativa-Idade/"
    "resultados/deepface"
)

PASTA_RESULTADOS_MIVOLO = (
    "/content/drive/MyDrive/Classroom/"
    "9923 31 - TRABALHO DE CONCLUSÃO DE CURSO 2026/"
    "DESENVOLVIMENTO/Ambientes Colab/"
    "resultados/MiVOLO"
)

# Diretório de saída do Notebook 05
PASTA_SAIDA = (
    "/content/drive/MyDrive/TCC-Estimativa-Idade/"
    "resultados/matrizes_confusao"
)

os.makedirs(PASTA_SAIDA, exist_ok=True)

# ------------------------------------------------------------
# Configurações metodológicas
# ------------------------------------------------------------

LIMIARES_PRINCIPAIS = [12, 16, 18]
LIMIARES_COMPLEMENTARES = [17]

LIMIARES = LIMIARES_PRINCIPAIS + LIMIARES_COMPLEMENTARES

FAIXAS_ETARIAS = [
    (0, 11, "0 a 11 anos"),
    (12, 15, "12 a 15 anos"),
    (16, 17, "16 a 17 anos"),
    (18, np.inf, "18 anos ou mais")
]

ROTULOS_FAIXAS = [
    faixa[2] for faixa in FAIXAS_ETARIAS
]

TECNOLOGIAS = {
    "DeepFace": {
        "pasta": PASTA_RESULTADOS_DEEPFACE,
        "status_sucesso": "Sucesso"
    },
    "MiVOLO": {
        "pasta": PASTA_RESULTADOS_MIVOLO,
        "status_sucesso": "OK"
    }
}

print("Configurações carregadas.")
print("Limiares:", LIMIARES)
print("Faixas etárias:", ROTULOS_FAIXAS)


In [ ]:

# ------------------------------------------------------------
# Identificar os arquivos CSV de cada tecnologia
# ------------------------------------------------------------

arquivos_por_tecnologia = {}

for tecnologia, configuracao in TECNOLOGIAS.items():

    pasta = configuracao["pasta"]

    arquivos = sorted(
        glob.glob(os.path.join(pasta, "*.csv"))
    )

    print("\n" + "=" * 70)
    print(tecnologia)
    print("=" * 70)

    print("Diretório:", pasta)
    print("Arquivos encontrados:", len(arquivos))

    for arquivo in arquivos:
        print("-", os.path.basename(arquivo))

    if len(arquivos) != 3:
        raise ValueError(
            f"Esperados 3 CSVs para {tecnologia}, "
            f"mas foram encontrados {len(arquivos)}."
        )

    arquivos_por_tecnologia[tecnologia] = arquivos

print("\nVerificação concluída.")


In [ ]:

def identificar_parte(nome_arquivo):

    nome = nome_arquivo.lower()

    if "part1" in nome or "parte1" in nome or "parte_1" in nome:
        return "Parte 1"

    if "part2" in nome or "parte2" in nome or "parte_2" in nome:
        return "Parte 2"

    if "part3" in nome or "parte3" in nome or "parte_3" in nome:
        return "Parte 3"

    return None


for tecnologia, arquivos in arquivos_por_tecnologia.items():

    partes_encontradas = [
        identificar_parte(os.path.basename(arquivo))
        for arquivo in arquivos
    ]

    if sorted(partes_encontradas, key=str) != [
        "Parte 1", "Parte 2", "Parte 3"
    ]:
        raise ValueError(
            f"Não foi possível identificar corretamente "
            f"as três partições de {tecnologia}: "
            f"{partes_encontradas}"
        )

print("Identificação das partições validada.")


In [ ]:

# ------------------------------------------------------------
# Funções auxiliares
# ------------------------------------------------------------

def converter_numerico(serie):

    return pd.to_numeric(
        serie,
        errors="coerce"
    )


def classificar_faixa(idade):

    if pd.isna(idade) or idade < 0:
        return None

    for minimo, maximo, rotulo in FAIXAS_ETARIAS:

        if minimo <= idade <= maximo:
            return rotulo

    return None


# ------------------------------------------------------------
# Carregar os arquivos
# ------------------------------------------------------------

dados = []

for tecnologia, arquivos in arquivos_por_tecnologia.items():

    status_sucesso = TECNOLOGIAS[tecnologia]["status_sucesso"]

    for caminho in arquivos:

        parte = identificar_parte(os.path.basename(caminho))

        df = pd.read_csv(caminho)

        colunas_obrigatorias = [
            "Arquivo",
            "Idade Real",
            "Idade Estimada",
            "Status"
        ]

        faltantes = [
            coluna for coluna in colunas_obrigatorias
            if coluna not in df.columns
        ]

        if faltantes:
            raise ValueError(
                f"Colunas ausentes em {caminho}: {faltantes}"
            )

        df["Tecnologia"] = tecnologia
        df["Parte"] = parte

        df["Idade Real"] = converter_numerico(
            df["Idade Real"]
        )

        df["Idade Estimada"] = converter_numerico(
            df["Idade Estimada"]
        )

        # Idade de referência válida
        df["Referencia_Valida"] = (
            df["Idade Real"].notna()
            & np.isfinite(df["Idade Real"])
            & (df["Idade Real"] >= 0)
        )

        # Status de inferência bem-sucedida
        df["Status_Sucesso"] = (
            df["Status"].astype(str).str.strip()
            == status_sucesso
        )

        # Saída numérica: sucesso e estimativa finita
        df["Saida_Numerica"] = (
            df["Status_Sucesso"]
            & df["Idade Estimada"].notna()
            & np.isfinite(df["Idade Estimada"])
        )

        # Decisão válida: estimativa finita e não negativa
        df["Decisao_Valida"] = (
            df["Saida_Numerica"]
            & (df["Idade Estimada"] >= 0)
        )

        # Classificação multiclasse
        df["Faixa_Real"] = df["Idade Real"].apply(
            classificar_faixa
        )

        df["Faixa_Estimada"] = df["Idade Estimada"].where(
            df["Decisao_Valida"]
        ).apply(classificar_faixa)

        dados.append(df)

# ------------------------------------------------------------
# Consolidar registros
# ------------------------------------------------------------

df_dados = pd.concat(
    dados,
    ignore_index=True
)

print("Registros carregados:", len(df_dados))

display(
    df_dados[
        [
            "Tecnologia",
            "Parte",
            "Arquivo",
            "Idade Real",
            "Idade Estimada",
            "Status",
            "Saida_Numerica",
            "Decisao_Valida",
            "Faixa_Real",
            "Faixa_Estimada"
        ]
    ].head()
)


In [ ]:

diagnostico = (
    df_dados
    .groupby(["Tecnologia", "Parte"], observed=True)
    .agg(
        Total_Registros=("Arquivo", "size"),
        Referencias_Validas=("Referencia_Valida", "sum"),
        Saidas_Numericas=("Saida_Numerica", "sum"),
        Decisoes_Validas=("Decisao_Valida", "sum")
    )
    .reset_index()
)

diagnostico["Estimativas_Negativas"] = (
    df_dados.assign(
        Estimativa_Negativa=(
            df_dados["Saida_Numerica"]
            & (df_dados["Idade Estimada"] < 0)
        )
    )
    .groupby(["Tecnologia", "Parte"], observed=True)[
        "Estimativa_Negativa"
    ]
    .sum()
    .values
)

diagnostico["Sem_Decisao_Valida"] = (
    diagnostico["Referencias_Validas"]
    - diagnostico["Decisoes_Validas"]
)

display(diagnostico)

print("\nTotais por tecnologia:")

display(
    diagnostico
    .groupby("Tecnologia", observed=True)
    .sum(numeric_only=True)
    .reset_index()
)


In [ ]:

## 1. Classificação multiclasse por faixas etárias

Nesta análise, as idades de referência e as idades estimadas são organizadas em quatro categorias:

- 0 a 11 anos;
- 12 a 15 anos;
- 16 a 17 anos;
- 18 anos ou mais.

As matrizes de confusão permitem observar a distribuição dos acertos e erros entre as faixas etárias.

São consideradas apenas decisões válidas do modelo. Os registros sem decisão válida são contabilizados separadamente, para que não sejam interpretados como classificações incorretas em uma faixa específica.

Além das matrizes, será calculado o percentual de acerto por faixa de referência, considerando tanto o total de registros daquela faixa quanto somente as decisões válidas.


In [ ]:

def calcular_matriz_multiclasse(df):

    base = df[
        df["Referencia_Valida"]
        & df["Decisao_Valida"]
        & df["Faixa_Real"].notna()
        & df["Faixa_Estimada"].notna()
    ].copy()

    matriz = confusion_matrix(
        base["Faixa_Real"],
        base["Faixa_Estimada"],
        labels=ROTULOS_FAIXAS
    )

    return matriz, base


In [ ]:

matrizes_multiclasse = {}

for tecnologia in TECNOLOGIAS:

    for parte in ["Parte 1", "Parte 2", "Parte 3"]:

        base = df_dados[
            (df_dados["Tecnologia"] == tecnologia)
            & (df_dados["Parte"] == parte)
        ]

        matriz, base_valida = calcular_matriz_multiclasse(base)

        matrizes_multiclasse[
            (tecnologia, parte)
        ] = matriz

        fig, ax = plt.subplots(figsize=(7, 6))

        display_matriz = ConfusionMatrixDisplay(
            confusion_matrix=matriz,
            display_labels=ROTULOS_FAIXAS
        )

        display_matriz.plot(
            ax=ax,
            values_format="d",
            cmap="Blues",
            xticks_rotation=30,
            colorbar=False
        )

        ax.set_title(
            f"{tecnologia} - {parte}\n"
            "Matriz de Confusão Multiclasse"
        )

        ax.set_xlabel("Faixa estimada")
        ax.set_ylabel("Faixa de referência")

        plt.tight_layout()

        nome = (
            f"matriz_multiclasse_{tecnologia.lower()}_"
            f"{parte.lower().replace(' ', '_')}.png"
        )

        plt.savefig(
            os.path.join(PASTA_SAIDA, nome),
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

print("Matrizes multiclasse por partição geradas.")


In [ ]:

for tecnologia in TECNOLOGIAS:

    base = df_dados[
        df_dados["Tecnologia"] == tecnologia
    ]

    matriz, base_valida = calcular_matriz_multiclasse(base)

    matrizes_multiclasse[
        (tecnologia, "Consolidado")
    ] = matriz

    fig, ax = plt.subplots(figsize=(7, 6))

    display_matriz = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=ROTULOS_FAIXAS
    )

    display_matriz.plot(
        ax=ax,
        values_format="d",
        cmap="Blues",
        xticks_rotation=30,
        colorbar=False
    )

    ax.set_title(
        f"{tecnologia} - UTKFace Consolidada\n"
        "Matriz de Confusão Multiclasse"
    )

    ax.set_xlabel("Faixa estimada")
    ax.set_ylabel("Faixa de referência")

    plt.tight_layout()

    nome = (
        f"matriz_multiclasse_{tecnologia.lower()}_"
        "consolidado.png"
    )

    plt.savefig(
        os.path.join(PASTA_SAIDA, nome),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

print("Matrizes multiclasse consolidadas geradas.")


In [ ]:

resultados_multiclasse = []

for tecnologia in TECNOLOGIAS:

    for parte in ["Parte 1", "Parte 2", "Parte 3", "Consolidado"]:

        if parte == "Consolidado":
            base = df_dados[
                df_dados["Tecnologia"] == tecnologia
            ]
        else:
            base = df_dados[
                (df_dados["Tecnologia"] == tecnologia)
                & (df_dados["Parte"] == parte)
            ]

        matriz = matrizes_multiclasse[
            (tecnologia, parte)
        ]

        for indice, faixa in enumerate(ROTULOS_FAIXAS):

            total_referencia = len(
                base[
                    base["Referencia_Valida"]
                    & (base["Faixa_Real"] == faixa)
                ]
            )

            base_faixa_valida = base[
                base["Referencia_Valida"]
                & base["Decisao_Valida"]
                & (base["Faixa_Real"] == faixa)
                & base["Faixa_Estimada"].notna()
            ]

            total_decisoes = len(base_faixa_valida)

            acertos = int(matriz[indice, indice])

            sem_decisao = (
                total_referencia - total_decisoes
            )

            acerto_total = (
                acertos / total_referencia * 100
                if total_referencia else np.nan
            )

            acerto_condicional = (
                acertos / total_decisoes * 100
                if total_decisoes else np.nan
            )

            resultados_multiclasse.append({

                "Tecnologia": tecnologia,
                "Parte": parte,
                "Faixa de Referência": faixa,
                "Total de Referência": total_referencia,
                "Decisões Válidas": total_decisoes,
                "Sem Decisão Válida": sem_decisao,
                "Acertos": acertos,
                "Acerto sobre Total (%)": acerto_total,
                "Acerto Condicional (%)": acerto_condicional

            })

df_resultados_multiclasse = pd.DataFrame(
    resultados_multiclasse
)

display(
    df_resultados_multiclasse.round(2)
)


In [ ]:

## 2. Classificação binária por limiares etários

A classificação binária avalia se o modelo consegue distinguir indivíduos abaixo de determinado limiar daqueles cuja idade de referência é igual ou superior a esse ponto de corte.

Serão avaliados os limiares de 12, 16 e 18 anos, definidos como principais, e o limiar complementar de 17 anos.

Para cada limiar:

- Classe 0: idade abaixo do limiar;
- Classe 1: idade igual ou superior ao limiar.

As matrizes de confusão serão calculadas considerando apenas decisões válidas. Os registros sem decisão válida serão apresentados separadamente.

Serão calculadas as métricas:

- Verdadeiros positivos (TP);
- Verdadeiros negativos (TN);
- Falsos positivos (FP);
- Falsos negativos (FN);
- Acurácia total;
- Acurácia condicional;
- Taxa de verdadeiros positivos (TPR);
- Taxa de verdadeiros negativos (TNR);
- Taxa de falsos positivos (FPR);
- Taxa de falsos negativos (FNR).


In [ ]:

def divisao_segura(numerador, denominador):

    if denominador == 0:
        return np.nan

    return numerador / denominador


def calcular_metricas_binarias(base, limiar):

    # Referências válidas
    referencias = base[
        base["Referencia_Valida"]
    ].copy()

    # Decisões válidas
    decisoes = referencias[
        referencias["Decisao_Valida"]
    ].copy()

    # Classes reais e estimadas
    y_real = (
        decisoes["Idade Real"] >= limiar
    ).astype(int)

    y_estimado = (
        decisoes["Idade Estimada"] >= limiar
    ).astype(int)

    matriz = confusion_matrix(
        y_real,
        y_estimado,
        labels=[0, 1]
    )

    tn, fp, fn, tp = matriz.ravel()

    total_referencia = len(referencias)
    total_decisoes = len(decisoes)
    sem_decisao = total_referencia - total_decisoes

    # Acurácia condicional:
    # somente decisões válidas
    acuracia_condicional = divisao_segura(
        tp + tn,
        total_decisoes
    ) * 100

    # Acurácia total:
    # decisões inválidas são contabilizadas como
    # ausência de acerto no denominador
    acuracia_total = divisao_segura(
        tp + tn,
        total_referencia
    ) * 100

    tpr = divisao_segura(
        tp,
        tp + fn
    ) * 100

    tnr = divisao_segura(
        tn,
        tn + fp
    ) * 100

    fpr = divisao_segura(
        fp,
        fp + tn
    ) * 100

    fnr = divisao_segura(
        fn,
        fn + tp
    ) * 100

    resultado = {

        "Total de Referência": total_referencia,
        "Decisões Válidas": total_decisoes,
        "Sem Decisão Válida": sem_decisao,

        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,

        "Acurácia Total (%)": acuracia_total,
        "Acurácia Condicional (%)": acuracia_condicional,

        "TPR (%)": tpr,
        "TNR (%)": tnr,
        "FPR (%)": fpr,
        "FNR (%)": fnr

    }

    return matriz, resultado


In [ ]:

resultados_binarios = []
matrizes_binarias = {}

for tecnologia in TECNOLOGIAS:

    for parte in ["Parte 1", "Parte 2", "Parte 3"]:

        base = df_dados[
            (df_dados["Tecnologia"] == tecnologia)
            & (df_dados["Parte"] == parte)
        ]

        for limiar in LIMIARES:

            matriz, metricas = calcular_metricas_binarias(
                base,
                limiar
            )

            matrizes_binarias[
                (tecnologia, parte, limiar)
            ] = matriz

            tipo_limiar = (
                "Principal"
                if limiar in LIMIARES_PRINCIPAIS
                else "Complementar"
            )

            resultados_binarios.append({

                "Tecnologia": tecnologia,
                "Parte": parte,
                "Limiar": limiar,
                "Tipo de Limiar": tipo_limiar,
                **metricas

            })

            fig, ax = plt.subplots(figsize=(5, 5))

            display_matriz = ConfusionMatrixDisplay(
                confusion_matrix=matriz,
                display_labels=[
                    f"< {limiar}",
                    f">= {limiar}"
                ]
            )

            display_matriz.plot(
                ax=ax,
                values_format="d",
                cmap="Blues",
                colorbar=False
            )

            ax.set_title(
                f"{tecnologia} - {parte}\n"
                f"Limiar de {limiar} anos"
            )

            ax.set_xlabel("Classe estimada")
            ax.set_ylabel("Classe de referência")

            plt.tight_layout()

            nome = (
                f"matriz_binaria_{tecnologia.lower()}_"
                f"{parte.lower().replace(' ', '_')}_"
                f"limiar_{limiar}.png"
            )

            plt.savefig(
                os.path.join(PASTA_SAIDA, nome),
                dpi=300,
                bbox_inches="tight"
            )

            plt.show()
            plt.close()

print("Matrizes binárias por partição geradas.")


In [ ]:

for tecnologia in TECNOLOGIAS:

    base = df_dados[
        df_dados["Tecnologia"] == tecnologia
    ]

    for limiar in LIMIARES:

        matriz, metricas = calcular_metricas_binarias(
            base,
            limiar
        )

        matrizes_binarias[
            (tecnologia, "Consolidado", limiar)
        ] = matriz

        tipo_limiar = (
            "Principal"
            if limiar in LIMIARES_PRINCIPAIS
            else "Complementar"
        )

        resultados_binarios.append({

            "Tecnologia": tecnologia,
            "Parte": "Consolidado",
            "Limiar": limiar,
            "Tipo de Limiar": tipo_limiar,
            **metricas

        })

        fig, ax = plt.subplots(figsize=(5, 5))

        display_matriz = ConfusionMatrixDisplay(
            confusion_matrix=matriz,
            display_labels=[
                f"< {limiar}",
                f">= {limiar}"
            ]
        )

        display_matriz.plot(
            ax=ax,
            values_format="d",
            cmap="Blues",
            colorbar=False
        )

        ax.set_title(
            f"{tecnologia} - UTKFace Consolidada\n"
            f"Limiar de {limiar} anos"
        )

        ax.set_xlabel("Classe estimada")
        ax.set_ylabel("Classe de referência")

        plt.tight_layout()

        nome = (
            f"matriz_binaria_{tecnologia.lower()}_"
            f"consolidado_limiar_{limiar}.png"
        )

        plt.savefig(
            os.path.join(PASTA_SAIDA, nome),
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

print("Matrizes binárias consolidadas geradas.")


In [ ]:

df_resultados_binarios = pd.DataFrame(
    resultados_binarios
)

ordem_partes = [
    "Parte 1",
    "Parte 2",
    "Parte 3",
    "Consolidado"
]

df_resultados_binarios["Parte"] = pd.Categorical(
    df_resultados_binarios["Parte"],
    categories=ordem_partes,
    ordered=True
)

df_resultados_binarios["Tecnologia"] = pd.Categorical(
    df_resultados_binarios["Tecnologia"],
    categories=["DeepFace", "MiVOLO"],
    ordered=True
)

df_resultados_binarios = (
    df_resultados_binarios
    .sort_values(
        ["Tecnologia", "Parte", "Limiar"]
    )
    .reset_index(drop=True)
)

colunas_percentuais = [
    "Acurácia Total (%)",
    "Acurácia Condicional (%)",
    "TPR (%)",
    "TNR (%)",
    "FPR (%)",
    "FNR (%)"
]

df_resultados_binarios[
    colunas_percentuais
] = df_resultados_binarios[
    colunas_percentuais
].round(2)

display(df_resultados_binarios)


In [ ]:

registros_matrizes_multiclasse = []

for (tecnologia, parte), matriz in matrizes_multiclasse.items():

    for i, faixa_real in enumerate(ROTULOS_FAIXAS):

        for j, faixa_estimada in enumerate(ROTULOS_FAIXAS):

            registros_matrizes_multiclasse.append({

                "Tecnologia": tecnologia,
                "Parte": parte,
                "Faixa de Referência": faixa_real,
                "Faixa Estimada": faixa_estimada,
                "Quantidade": int(matriz[i, j])

            })

df_matrizes_multiclasse = pd.DataFrame(
    registros_matrizes_multiclasse
)

display(df_matrizes_multiclasse)


In [ ]:

registros_matrizes_binarias = []

for (tecnologia, parte, limiar), matriz in matrizes_binarias.items():

    tn, fp, fn, tp = matriz.ravel()

    registros_matrizes_binarias.append({

        "Tecnologia": tecnologia,
        "Parte": parte,
        "Limiar": limiar,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)

    })

df_matrizes_binarias = pd.DataFrame(
    registros_matrizes_binarias
)

display(
    df_matrizes_binarias.sort_values(
        ["Tecnologia", "Parte", "Limiar"]
    )
)


In [ ]:

# ------------------------------------------------------------
# Exportar tabelas
# ------------------------------------------------------------

arquivos_exportados = {

    "resultados_multiclasse": (
        "resultados_classificacao_multiclasse_utkface.csv"
    ),

    "matrizes_multiclasse": (
        "matrizes_confusao_multiclasse_utkface.csv"
    ),

    "resultados_binarios": (
        "resultados_classificacao_binaria_limiares_utkface.csv"
    ),

    "matrizes_binarias": (
        "matrizes_confusao_binarias_utkface.csv"
    ),

    "diagnostico": (
        "diagnostico_decisoes_utkface.csv"
    )

}

df_resultados_multiclasse.to_csv(
    os.path.join(PASTA_SAIDA, arquivos_exportados["resultados_multiclasse"]),
    index=False,
    encoding="utf-8-sig"
)

df_resultados_multiclasse.to_csv(
    os.path.join(PASTA_SAIDA, arquivos_exportados["acerto_por_faixa"]),
    index=False,
    encoding="utf-8-sig"
)

df_matrizes_multiclasse.to_csv(
    os.path.join(PASTA_SAIDA, arquivos_exportados["matrizes_multiclasse"]),
    index=False,
    encoding="utf-8-sig"
)

df_resultados_binarios.to_csv(
    os.path.join(PASTA_SAIDA, arquivos_exportados["resultados_binarios"]),
    index=False,
    encoding="utf-8-sig"
)

df_matrizes_binarias.to_csv(
    os.path.join(PASTA_SAIDA, arquivos_exportados["matrizes_binarias"]),
    index=False,
    encoding="utf-8-sig"
)

diagnostico.to_csv(
    os.path.join(PASTA_SAIDA, arquivos_exportados["diagnostico"]),
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos exportados:")
for nome in arquivos_exportados.values():
    print("-", os.path.join(PASTA_SAIDA, nome))


In [ ]:

print("=" * 80)
print("VERIFICAÇÃO FINAL — NOTEBOOK 05")
print("=" * 80)

print("\n1. Registros carregados:")
print(
    df_dados.groupby("Tecnologia", observed=True)
    .size()
)

print("\n2. Matrizes multiclasse geradas:")
print(len(matrizes_multiclasse))

print("\n3. Matrizes binárias geradas:")
print(len(matrizes_binarias))

print("\n4. Limiares avaliados:")
print(LIMIARES)

print("\n5. Faixas etárias:")
for faixa in ROTULOS_FAIXAS:
    print("-", faixa)

print("\n6. Diretório de saída:")
print(PASTA_SAIDA)

print("\nPROCESSAMENTO FINALIZADO.")


In [ ]:

## 3. Critérios de interpretação dos resultados

As matrizes de confusão permitem identificar a distribuição dos acertos e erros dos modelos nas classificações multiclasse e binária.

Na classificação multiclasse, os valores da diagonal principal representam as classificações corretas, enquanto os valores fora da diagonal representam classificações em faixas diferentes da faixa de referência.

Na classificação binária, são considerados:

- **TP (Verdadeiros Positivos):** registros cuja idade de referência e estimativa pertencem à classe positiva.
- **TN (Verdadeiros Negativos):** registros cuja idade de referência e estimativa pertencem à classe negativa.
- **FP (Falsos Positivos):** registros abaixo do limiar na referência, mas classificados acima ou no limiar pelo modelo.
- **FN (Falsos Negativos):** registros acima ou no limiar na referência, mas classificados abaixo pelo modelo.

A acurácia total considera todos os registros com idade de referência válida no denominador, incluindo aqueles sem decisão válida. A acurácia condicional considera somente as decisões válidas.

As taxas TPR, TNR, FPR e FNR são calculadas sobre as decisões válidas, permitindo analisar o comportamento do modelo quando efetivamente produz uma estimativa utilizável.

As estimativas negativas e os registros sem inferência bem-sucedida não são convertidos em classificações. Eles permanecem identificados no diagnóstico e são considerados na avaliação da cobertura e da acurácia total.

O limiar de 17 anos possui caráter complementar, enquanto os limiares de 12, 16 e 18 anos constituem os pontos de corte principais da análise.
